# AI + Data: Make Data Intelligent — E-Commerce Data Analysis

**Dataset:** AI + Data Masterclass Practice Dataset  
**Objective:** Understand data quality, clean the e-commerce data, analyze business performance, visualize insights, and export a clean dataset.

This notebook is based on the uploaded Excel file and its `Raw_Data` sheet.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

FILE_PATH = "AI + Data_ Make Data Intelligent _ Masterclass 1 _ Practice Dataset (1).xlsx"

df = pd.read_excel(FILE_PATH, sheet_name="Raw_Data")

print("Dataset loaded successfully")
print("Shape:", df.shape)
display(df.head())


## 1. Understand the dataset

In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.to_frame("Data Type"))

print("\nDataset information:")
df.info()

print("\nStatistical summary:")
display(df.describe(include="all").T)


## 2. Missing values and duplicate records

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
print("Missing values:")
display(missing.to_frame("Missing Count"))

duplicates = df.duplicated().sum()
print("Duplicate records:", duplicates)

if duplicates:
    display(df[df.duplicated(keep=False)])


## 3. Identify inconsistent categorical values

In [ ]:
for col in ["Category", "Region", "Product"]:
    print(f"\nUnique values in {col}:")
    print(df[col].dropna().unique())
    print("\nValue counts:")
    display(df[col].value_counts(dropna=False))


## 4. Validate dates, quantity, revenue and profit

In [ ]:
# Date validation
date_check = pd.to_datetime(df["Order_Date"], errors="coerce", dayfirst=True)

print("Invalid/unparseable dates:", date_check.isna().sum())
display(df.loc[date_check.isna(), ["Order_ID", "Order_Date"]])

# Quantity validation
quantity_numeric = pd.to_numeric(df["Quantity"], errors="coerce")
print("\nInvalid quantity values:", quantity_numeric.isna().sum())
print("Negative quantity records:", (quantity_numeric < 0).sum())
display(df.loc[quantity_numeric < 0, ["Order_ID", "Product", "Quantity"]])

# Revenue validation
revenue_clean_check = (
    df["Revenue"].astype(str)
    .str.replace("₹", "", regex=False)
    .str.replace("â‚¹", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)
revenue_numeric = pd.to_numeric(revenue_clean_check, errors="coerce")

print("\nInvalid revenue values:", revenue_numeric.isna().sum())
display(df.loc[revenue_numeric.isna(), ["Order_ID", "Revenue"]])

# Profit validation
profit_numeric = pd.to_numeric(df["Profit"], errors="coerce")
print("Invalid profit values:", profit_numeric.isna().sum())


## 5. Clean the data

In [ ]:
clean_df = df.copy()

# Standardize category and region text
clean_df["Category"] = (
    clean_df["Category"]
    .astype("string")
    .str.strip()
    .str.title()
)

clean_df["Region"] = (
    clean_df["Region"]
    .astype("string")
    .str.strip()
    .str.title()
)

clean_df["Product"] = (
    clean_df["Product"]
    .astype("string")
    .str.strip()
)

# Parse dates; invalid dates become NaT
clean_df["Order_Date"] = pd.to_datetime(
    clean_df["Order_Date"],
    errors="coerce",
    dayfirst=True
)

# Numeric conversion
clean_df["Quantity"] = pd.to_numeric(
    clean_df["Quantity"], errors="coerce"
)

clean_df["Revenue"] = (
    clean_df["Revenue"].astype(str)
    .str.replace("₹", "", regex=False)
    .str.replace("â‚¹", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)
clean_df["Revenue"] = pd.to_numeric(
    clean_df["Revenue"], errors="coerce"
)

clean_df["Profit"] = pd.to_numeric(
    clean_df["Profit"], errors="coerce"
)

# Negative quantity is invalid; treat it as missing
clean_df.loc[clean_df["Quantity"] < 0, "Quantity"] = np.nan

# Remove exact duplicates
before = len(clean_df)
clean_df = clean_df.drop_duplicates().reset_index(drop=True)
print("Duplicates removed:", before - len(clean_df))

# Fill numeric missing values using medians
for col in ["Quantity", "Revenue", "Profit"]:
    clean_df[col] = clean_df[col].fillna(clean_df[col].median())

# Fill missing category/region/product values
for col in ["Category", "Region", "Product"]:
    clean_df[col] = clean_df[col].fillna("Unknown")

print("Cleaned shape:", clean_df.shape)
display(clean_df.head())


## 6. Data-quality check after cleaning

In [ ]:
print("Missing values after cleaning:")
display(clean_df.isnull().sum().to_frame("Missing Count"))

print("Duplicate records after cleaning:", clean_df.duplicated().sum())

print("\nCleaned data types:")
display(clean_df.dtypes.to_frame("Data Type"))


## 7. Overall business KPIs

In [ ]:
total_revenue = clean_df["Revenue"].sum()
total_profit = clean_df["Profit"].sum()
total_quantity = clean_df["Quantity"].sum()
total_orders = clean_df["Order_ID"].nunique()

profit_margin = (total_profit / total_revenue * 100) if total_revenue else 0

kpis = pd.DataFrame({
    "KPI": [
        "Total Orders",
        "Total Quantity Sold",
        "Total Revenue",
        "Total Profit",
        "Profit Margin (%)"
    ],
    "Value": [
        total_orders,
        total_quantity,
        total_revenue,
        total_profit,
        profit_margin
    ]
})

display(kpis)


## 8. Revenue and profit by category

In [ ]:
category_summary = (
    clean_df.groupby("Category")
    .agg(
        Orders=("Order_ID", "nunique"),
        Quantity=("Quantity", "sum"),
        Revenue=("Revenue", "sum"),
        Profit=("Profit", "sum")
    )
    .sort_values("Revenue", ascending=False)
)

category_summary["Profit_Margin_%"] = (
    category_summary["Profit"] /
    category_summary["Revenue"] * 100
)

display(category_summary.round(2))


In [ ]:
plt.figure(figsize=(8, 5))
category_summary["Revenue"].plot(kind="bar")
plt.title("Revenue by Category")
plt.xlabel("Category")
plt.ylabel("Revenue")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
category_summary["Profit"].plot(kind="bar")
plt.title("Profit by Category")
plt.xlabel("Category")
plt.ylabel("Profit")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


## 9. Revenue and profit by region

In [ ]:
region_summary = (
    clean_df.groupby("Region")
    .agg(
        Orders=("Order_ID", "nunique"),
        Quantity=("Quantity", "sum"),
        Revenue=("Revenue", "sum"),
        Profit=("Profit", "sum")
    )
    .sort_values("Revenue", ascending=False)
)

region_summary["Profit_Margin_%"] = (
    region_summary["Profit"] /
    region_summary["Revenue"] * 100
)

display(region_summary.round(2))


In [ ]:
plt.figure(figsize=(8, 5))
region_summary["Revenue"].plot(kind="bar")
plt.title("Revenue by Region")
plt.xlabel("Region")
plt.ylabel("Revenue")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
region_summary["Profit"].plot(kind="bar")
plt.title("Profit by Region")
plt.xlabel("Region")
plt.ylabel("Profit")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


## 10. Top products

In [ ]:
product_summary = (
    clean_df.groupby("Product")
    .agg(
        Orders=("Order_ID", "nunique"),
        Quantity=("Quantity", "sum"),
        Revenue=("Revenue", "sum"),
        Profit=("Profit", "sum")
    )
    .sort_values("Revenue", ascending=False)
)

print("Top 10 products by revenue:")
display(product_summary.head(10).round(2))


In [ ]:
top10 = product_summary.head(10).sort_values("Revenue")

plt.figure(figsize=(10, 6))
top10["Revenue"].plot(kind="barh")
plt.title("Top 10 Products by Revenue")
plt.xlabel("Revenue")
plt.ylabel("Product")
plt.tight_layout()
plt.show()


## 11. Monthly sales trend

In [ ]:
monthly_summary = (
    clean_df.dropna(subset=["Order_Date"])
    .assign(Month=lambda x: x["Order_Date"].dt.to_period("M").astype(str))
    .groupby("Month")
    .agg(
        Orders=("Order_ID", "nunique"),
        Revenue=("Revenue", "sum"),
        Profit=("Profit", "sum")
    )
    .sort_index()
)

display(monthly_summary.round(2))


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(
    monthly_summary.index,
    monthly_summary["Revenue"],
    marker="o"
)
plt.title("Monthly Revenue Trend")
plt.xlabel("Month")
plt.ylabel("Revenue")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 12. Business insights

In [ ]:
best_category = category_summary["Revenue"].idxmax()
best_region = region_summary["Revenue"].idxmax()
best_product = product_summary["Revenue"].idxmax()
most_profitable_category = category_summary["Profit"].idxmax()
most_profitable_region = region_summary["Profit"].idxmax()

print("BUSINESS INSIGHTS")
print("=" * 60)
print(f"1. Highest-revenue category: {best_category}")
print(f"2. Highest-revenue region: {best_region}")
print(f"3. Highest-revenue product: {best_product}")
print(f"4. Most profitable category: {most_profitable_category}")
print(f"5. Most profitable region: {most_profitable_region}")
print(f"6. Overall profit margin: {profit_margin:.2f}%")


## 13. Final cleaned dataset

The cleaned dataset is exported as an Excel file for further use.


In [ ]:
OUTPUT_FILE = "Cleaned_Ecommerce_Data.xlsx"

clean_df.to_excel(
    OUTPUT_FILE,
    index=False
)

print(f"Cleaned file saved as: {OUTPUT_FILE}")


## Conclusion

The analysis identifies data-quality issues, standardizes inconsistent values, handles invalid numeric/date values, calculates business KPIs, compares category and regional performance, identifies top products, and examines the monthly revenue trend.

The resulting `Cleaned_Ecommerce_Data.xlsx` can be used as the cleaned output dataset.
